In [ ]:
"""
Ensemble Machine Learning for CLAMP-based Paleoclimate Reconstruction

This script implements the methodology described in the manuscript:
- Reads modern calibration data (393 sites) and fossil data (73 assemblages)
- Trains five models: Random Forest, AdaBoost, XGBoost, CatBoost, and a Stacking ensemble
- Evaluates performance via repeated 10-fold cross-validation (R², RMSE, MAE, MAPE)
- Interprets feature importance using SHAP
- Reconstructs paleoclimate from fossil leaf assemblages

Author: Based on provided methods
Date: 2025 (modified to output SVG figures)
"""

import numpy as np
import pandas as pd
import warnings
from sklearn.base import clone
warnings.filterwarnings('ignore')

# Machine learning libraries
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import RepeatedKFold, cross_val_predict, cross_validate
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

import xgboost as xgb
import catboost as cb
import shap

# ----------------------------------------------------------------------
# 1. Load datasets
# ----------------------------------------------------------------------
file_path = "Global378+Tibet15+Fossil73_更正.xlsx"

# Modern calibration dataset (393 sites: 378 global + 15 Tibetan Plateau)
df_modern = pd.read_excel(file_path, sheet_name="plants")
# Fossil dataset (73 assemblages)
df_fossil = pd.read_excel(file_path, sheet_name="fossils")

print(f"Modern dataset shape: {df_modern.shape}")
print(f"Fossil dataset shape: {df_fossil.shape}")

# ----------------------------------------------------------------------
# 2. Define leaf physiognomic traits (31 features) and climate targets (7 variables)
# ----------------------------------------------------------------------
feature_cols = [
    'Lobed', 'No Teeth', 'Teeth Regul', 'Teeth Close', 'Teeth Round', 'Teeth Acute', 'Teeth Compo',
    'Nanophyl', 'Leptophyl', 'Leptophyl 2', 'Microphyl 1', 'Microphyl 2', 'Microphyl 3',
    'Mesophyl 1', 'Mesophyl 2', 'Mesophyl 3', 'Apex Emargi', 'Apex Round', 'Apex Acute', 'Apex Attenu',
    'Base Cordat', 'Base Round', 'Base Acute', 'L:W <1:1', 'L:W 1-2:1', 'L:W 2-3:1', 'L:W 3-4:1',
    'L:W >4:1', 'Obovate', 'Elliptic', 'Ovate'
]

target_cols = ['MAT', 'WMMT', 'CMMT', 'GROWSEAS', 'RH', 'SH', 'ENTHAL']

# Check availability
missing_features = [c for c in feature_cols if c not in df_modern.columns]
missing_targets = [c for c in target_cols if c not in df_modern.columns]
if missing_features:
    raise KeyError(f"Missing features: {missing_features}")
if missing_targets:
    raise KeyError(f"Missing targets: {missing_targets}")

# ----------------------------------------------------------------------
# 3. Data cleaning (drop any rows with missing values)
# ----------------------------------------------------------------------
modern_features = df_modern[feature_cols].copy()
modern_targets = df_modern[target_cols].copy()
mask = modern_features.isna().any(axis=1) | modern_targets.isna().any(axis=1)
if mask.any():
    print(f"Dropping {mask.sum()} rows with missing values in modern dataset.")
    modern_features = modern_features[~mask]
    modern_targets = modern_targets[~mask]

print(f"Clean modern dataset: {modern_features.shape[0]} samples")

# For fossil data: keep only available features (should be same columns)
fossil_features = df_fossil[feature_cols].copy()
if fossil_features.isna().any().any():
    print("Warning: Fossil dataset contains missing values. Filling with column median.")
    fossil_features = fossil_features.fillna(fossil_features.median())

# ----------------------------------------------------------------------
# 4. Define evaluation metrics
# ----------------------------------------------------------------------
def mean_absolute_percentage_error(y_true, y_pred, eps=1e-6):
    """Mean absolute percentage error, safe for near-zero values."""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    denominator = np.abs(y_true) + eps
    return np.mean(np.abs((y_true - y_pred) / denominator)) * 100

def evaluate_model(y_true, y_pred):
    """Compute R², RMSE, MAE, MAPE."""
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return {'R2': r2, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

# ----------------------------------------------------------------------
# 5. Define models (with GPU acceleration for XGBoost and CatBoost)
# ----------------------------------------------------------------------
# Base models (using default hyperparameters for reproducibility)
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
ada = AdaBoostRegressor(n_estimators=200, random_state=42)

# XGBoost with GPU
xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    random_state=42,
    verbosity=0,
    tree_method= 'hist',           # 新版本推荐使用 hist 配合
    device='cuda',                # 或者 'cuda:0'
    predictor= 'auto'
)

# CatBoost with GPU
cat = cb.CatBoostRegressor(
    n_estimators=200,
    random_seed=42,
    verbose=False,
    task_type='CPU',
    devices='0'
)

# Stacking ensemble (base learners + linear meta-learner)
stack = StackingRegressor(
    estimators=[('rf', rf), ('ada', ada), ('xgb', xgb_model), ('cat', cat)],
    final_estimator=LinearRegression(),
    cv=5
)

models = {
    'Random Forest': rf,
    'AdaBoost': ada,
    'XGBoost': xgb_model,
    'CatBoost': cat,
    'Stacking': stack
}
# ----------------------------------------------------------------------
# 6. Cross-validation setup
# ----------------------------------------------------------------------

from sklearn.model_selection import KFold

results = {
    target: {
        name: {
            'R2': [],
            'RMSE': [],
            'MAE': [],
            'MAPE': []
        }
        for name in models
    }
    for target in target_cols
}

print("\nStarting repeated 10-fold cross-validation (5 repeats)...")

for target in target_cols:

    y = modern_targets[target].values
    X = modern_features.values

    for name, model in models.items():

        r2_scores = []
        rmse_scores = []
        mae_scores = []
        mape_scores = []

        # 5 repeats
        for repeat in range(5):

            kf = KFold(
                n_splits=10,
                shuffle=True,
                random_state=42 + repeat
            )

            predictions = []
            true_values = []

            for train_idx, test_idx in kf.split(X):

                X_train = X[train_idx]
                X_test = X[test_idx]

                y_train = y[train_idx]
                y_test = y[test_idx]

                model_clone = clone(model)

                model_clone.fit(
                    X_train,
                    y_train
                )

                y_pred = model_clone.predict(
                    X_test
                )

                predictions.extend(y_pred)
                true_values.extend(y_test)

            scores = evaluate_model(
                true_values,
                predictions
            )

            r2_scores.append(scores['R2'])
            rmse_scores.append(scores['RMSE'])
            mae_scores.append(scores['MAE'])
            mape_scores.append(scores['MAPE'])

        results[target][name]['R2'] = r2_scores
        results[target][name]['RMSE'] = rmse_scores
        results[target][name]['MAE'] = mae_scores
        results[target][name]['MAPE'] = mape_scores

    print(f"Finished {target}")
# Aggregate results across folds/repeats and compute mean ± std
summary = {}
for target in target_cols:
    summary[target] = {}
    for name in models:
        summary[target][name] = {}
        for metric in ['R2', 'RMSE', 'MAE', 'MAPE']:
            mean_val = np.mean(results[target][name][metric])
            std_val = np.std(results[target][name][metric])
            summary[target][name][metric] = (mean_val, std_val)

# Print performance table
print("\n" + "="*80)
print("MODEL PERFORMANCE (mean ± std across 5×10-fold CV)")
print("="*80)
for target in target_cols:
    print(f"\n{target}:")
    print(f"{'Model':<15} {'R²':<12} {'RMSE':<12} {'MAE':<12} {'MAPE(%)':<12}")
    print("-"*60)
    for name in models:
        r2 = f"{summary[target][name]['R2'][0]:.3f}±{summary[target][name]['R2'][1]:.3f}"
        rmse = f"{summary[target][name]['RMSE'][0]:.2f}±{summary[target][name]['RMSE'][1]:.2f}"
        mae = f"{summary[target][name]['MAE'][0]:.2f}±{summary[target][name]['MAE'][1]:.2f}"
        mape = f"{summary[target][name]['MAPE'][0]:.1f}±{summary[target][name]['MAPE'][1]:.1f}"
        print(f"{name:<15} {r2:<12} {rmse:<12} {mae:<12} {mape:<12}")

# Determine best model per target (based on highest R²)
best_models = {}
for target in target_cols:
    best_name = max(models.keys(), key=lambda n: summary[target][n]['R2'][0])
    best_models[target] = best_name
    print(f"\nBest model for {target}: {best_name} (R² = {summary[target][best_name]['R2'][0]:.3f})")
# ----------------------------------------------------------------------
# 7. Final training on full modern dataset and fossil prediction + SAVE MODELS
# ----------------------------------------------------------------------
from sklearn.base import clone
import joblib
import os
import json

print("\n" + "="*80)
print("FINAL TRAINING ON FULL MODERN DATASET & FOSSIL PREDICTION")
print("="*80)

model_dir = "./saved_models"
os.makedirs(model_dir, exist_ok=True)

fossil_predictions = pd.DataFrame(index=df_fossil.index)
fossil_predictions["Sample"] = df_fossil["Sample"]

best_model_info = {}

X_full = modern_features.values

for target in target_cols:

    print(f"\nTraining final model for {target}")

    y_full = modern_targets[target].values

    best_name = best_models[target]

    # 关键：重新克隆一个全新的模型
    model_final = clone(models[best_name])

    # 用全部现代样本重新训练
    model_final.fit(X_full, y_full)

    # 保存化石预测
    fossil_predictions[target] = model_final.predict(
        fossil_features.values
    )

    # 保存模型
    save_path = os.path.join(
        model_dir,
        f"{target}_{best_name}.joblib"
    )

    joblib.dump(model_final, save_path)

    best_model_info[target] = best_name

    print(
        f"Saved: {target}_{best_name}.joblib"
    )

# 保存预测结果
fossil_predictions.to_csv(
    "fossil_paleoclimate_predictions.csv",
    index=False
)

# 保存最佳模型映射
with open(
    os.path.join(model_dir, "best_models.json"),
    "w"
) as f:
    json.dump(best_model_info, f, indent=4)

print("\nAll models saved successfully.")
